# Stage 1 model

In [1]:
import pandas as pd
import plotly.express as px

from rich import print

from ambdes import Model, Runner, SimConfig, ArrivalConfig, TimesConfig

## Run model for 100 minutes with log messages

TODO: Convert this into logging tests which compare patient results with record in the vidigi logger.

In [2]:
arrival_config = ArrivalConfig(arrival_csv="data/arrivals.csv")
display(arrival_config.arrival_df)
display(arrival_config.nspp_df)

,C1,C2,C3,C4
monday,25,310,180,40
tuesday,24,295,170,38
wednesday,24,295,170,38
thursday,24,295,170,38
friday,25,305,178,40
saturday,28,340,200,45
sunday,27,330,195,43


,t,mean_iat
0,0,2.594595
1,1440,2.732448
2,2880,2.732448
3,4320,2.732448
4,5760,2.627737
5,7200,2.349103
6,8640,2.420168


In [3]:
times_config = TimesConfig(times_csv="data/times.csv")

Inspect this to see if times actually vary by response category:

In [4]:
times_config.times_df

,time,type,C1,C2,C3,C4
0,travel_to_scene,mean,8,10,12,12
1,travel_to_scene,sd,5,5,5,5
2,on_scene,mean,44,46,48,50
3,on_scene,sd,5,5,5,5
4,travel_to_hospital,mean,8,10,12,12
5,travel_to_hospital,sd,5,5,5,5
6,handover,mean,15,22,40,45
7,handover,sd,5,5,5,5
8,wrap_up,mean,5,5,5,5
9,wrap_up,sd,2,2,2,2


In [5]:
times_config.lognormal_config("travel_to_scene")

{'C1': {'class_name': 'Lognormal', 'params': {'mean': 8, 'stdev': 5}},
 'C2': {'class_name': 'Lognormal', 'params': {'mean': 10, 'stdev': 5}},
 'C3': {'class_name': 'Lognormal', 'params': {'mean': 12, 'stdev': 5}},
 'C4': {'class_name': 'Lognormal', 'params': {'mean': 12, 'stdev': 5}}}

In [6]:
config = SimConfig(
    arrival_config=arrival_config,
    times_config=times_config
)
config.n_ambulances = 1

In [7]:
print(config.dist_config)

{
    'call_arrival': {
        'class_name': 'NSPPThinning',
        'params': {
            'data':       t  mean_iat
0     0  2.594595
1  1440  2.732448
2  2880  2.732448
3  4320  2.732448
4  5760  2.627737
5  7200  2.349103
6  8640  2.420168
        }
    },
    'call_category': {
        'class_name': 'DiscreteEmpirical',
        'params': {
            'values': Index(['C1', 'C2', 'C3', 'C4'], dtype='object'),
            'freq': array([0.04547757, 0.5576737 , 0.32441131, 0.07243742])
        }
    },
    'time_to_scene': {
        'C1': {'class_name': 'Lognormal', 'params': {'mean': 8, 'stdev': 5}},
        'C2': {'class_name': 'Lognormal', 'params': {'mean': 10, 'stdev': 5}},
        'C3': {'class_name': 'Lognormal', 'params': {'mean': 12, 'stdev': 5}},
        'C4': {'class_name': 'Lognormal', 'params': {'mean': 12, 'stdev': 5}}
    },
    'on_scene_time': {
        'C1': {'class_name': 'Lognormal', 'params': {'mean': 44, 'stdev': 5}},
        'C2': {'class_name': 'Lognormal', 'params': {'mean': 46, 'stdev': 5}},
        'C3': {'class_name': 'Lognormal', 'params': {'mean': 48, 'stdev': 5}},
        'C4': {'class_name': 'Lognormal', 'params': {'mean': 50, 'stdev': 5}}
    },
    'time_to_hospital': {
        'C1': {'class_name': 'Lognormal', 'params': {'mean': 8, 'stdev': 5}},
        'C2': {'class_name': 'Lognormal', 'params': {'mean': 10, 'stdev': 5}},
        'C3': {'class_name': 'Lognormal', 'params': {'mean': 12, 'stdev': 5}},
        'C4': {'class_name': 'Lognormal', 'params': {'mean': 12, 'stdev': 5}}
    },
    'handover_time': {
        'C1': {'class_name': 'Lognormal', 'params': {'mean': 15, 'stdev': 5}},
        'C2': {'class_name': 'Lognormal', 'params': {'mean': 22, 'stdev': 5}},
        'C3': {'class_name': 'Lognormal', 'params': {'mean': 40, 'stdev': 5}},
        'C4': {'class_name': 'Lognormal', 'params': {'mean': 45, 'stdev': 5}}
    },
    'wrap_up_time': {
        'C1': {'class_name': 'Lognormal', 'params': {'mean': 5, 'stdev': 2}},
        'C2': {'class_name': 'Lognormal', 'params': {'mean': 5, 'stdev': 2}},
        'C3': {'class_name': 'Lognormal', 'params': {'mean': 5, 'stdev': 2}},
        'C4': {'class_name': 'Lognormal', 'params': {'mean': 5, 'stdev': 2}}
    }
}

In [8]:
model = Model(run_number=0, config=config)
print(model.dists)

{
    'call_arrival': NSPPThinning(data=      t  mean_iat
0     0  2.594595
1  1440  2.732448
2  2880  2.732448
3  4320  2.732448
4  5760  2..., interval=1440.0),
    'call_category': Discrete(values=[C1, C2, C3, ...], freq=[0.04547757478889612, 0.5576737006163818, 
0.32441130636504795, ...]),
    'handover_time': {
        'C1': Lognormal(mean=15, stdev=5),
        'C2': Lognormal(mean=22, stdev=5),
        'C3': Lognormal(mean=40, stdev=5),
        'C4': Lognormal(mean=45, stdev=5)
    },
    'on_scene_time': {
        'C1': Lognormal(mean=44, stdev=5),
        'C2': Lognormal(mean=46, stdev=5),
        'C3': Lognormal(mean=48, stdev=5),
        'C4': Lognormal(mean=50, stdev=5)
    },
    'time_to_hospital': {
        'C1': Lognormal(mean=8, stdev=5),
        'C2': Lognormal(mean=10, stdev=5),
        'C3': Lognormal(mean=12, stdev=5),
        'C4': Lognormal(mean=12, stdev=5)
    },
    'time_to_scene': {
        'C1': Lognormal(mean=8, stdev=5),
        'C2': Lognormal(mean=10, stdev=5),
        'C3': Lognormal(mean=12, stdev=5),
        'C4': Lognormal(mean=12, stdev=5)
    },
    'wrap_up_time': {
        'C1': Lognormal(mean=5, stdev=2),
        'C2': Lognormal(mean=5, stdev=2),
        'C3': Lognormal(mean=5, stdev=2),
        'C4': Lognormal(mean=5, stdev=2)
    }
}

In [9]:
model.dists["time_to_scene"]["C1"].sample()

3.940894139077947

In [10]:
model.run()

In [11]:
log = model.logger.to_dataframe()

In [12]:
print(model.patients[0].__dict__)
log[log["entity_id"] == 1]

{'patient_id': 1, 'category': 'C3', 'call_timestamp': 103.01102191839185, 'response_time': None}

,entity_id,event_type,event,time,run_number,resource_id
0,1,arrival_departure,arrival,6.610758,0,NaN
1,1,queue,ambulance_wait_begins,6.610758,0,NaN
2,1,resource_use,ambulance_assigned,6.610758,0,1.0
47,1,arrival_departure,arrival,103.011022,0,NaN
48,1,queue,ambulance_wait_begins,103.011022,0,NaN
79,1,resource_use_end,ambulance_available,127.576318,0,1.0
80,1,arrival_departure,depart,127.576318,0,NaN


In [13]:
print(model.patients[1].__dict__)
log[log["entity_id"] == 2]

{'patient_id': 2, 'category': 'C2', 'call_timestamp': 106.01208296727211, 'response_time': None}

,entity_id,event_type,event,time,run_number,resource_id
3,2,arrival_departure,arrival,13.078265,0,NaN
4,2,queue,ambulance_wait_begins,13.078265,0,NaN
49,2,arrival_departure,arrival,106.012083,0,NaN
50,2,queue,ambulance_wait_begins,106.012083,0,NaN
81,2,resource_use,ambulance_assigned,127.576318,0,1.0


## Run model for longer and inspect patient times

In [14]:
config = SimConfig(
    arrival_config=arrival_config,
    times_config=times_config,
    warm_up_period=0,
    data_collection_period=10080,  # One week
)
model = Model(run_number=0, config=config)
model.run()

In [15]:
df = pd.DataFrame(
    {
        "response_time": [p.response_time for p in model.patients],
        "category": [p.category for p in model.patients],
    }
)

fig = px.histogram(
    df,
    x="response_time",
    facet_col="category",
    nbins=50,
    category_orders={"category": ["C1", "C2", "C3", "C4"]},
    labels={
        "response_time": "Response time (minutes)",
        "category": "Category",
    },
    title="Distribution of response times by category",
)

# fig.update_yaxes(
#    matches=None,
#    showticklabels=True,
# )
fig.layout.yaxis.title.text = "Number of patients"

fig.show()

In [16]:
for cat in ["C1", "C2", "C3", "C4"]:
    fig = px.histogram(
        df[df["category"] == cat],
        x="response_time",
        nbins=20,
        title=f"Response times: {cat}",
        labels={"response_time": "Response time (minutes)"},
    )
    fig.update_yaxes(title_text="Number of patients")
    fig.show()

## Average results

In [17]:
runner = Runner(config)

In [18]:
results = runner.run_reps()

/home/amy/Documents/ambulance/ambdes/src/ambdes/results.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([summary, utilisation], ignore_index=True)
/home/amy/Documents/ambulance/ambdes/src/ambdes/results.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([summary, utilisation], ignore_index=True)
/home/amy/Documents/ambulance/ambdes/src/ambdes/results.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecate

In [19]:
results["patients"]

,run,patient_id,category,call_timestamp,response_time
0,0,1,C3,0.760945,7.500451
1,0,2,C2,1.129902,6.947597
2,0,3,C3,1.688854,9.439192
3,0,4,C2,2.833587,13.586722
4,0,5,C3,6.235100,17.280425
...,...,...,...,...,...
19662,4,3936,C2,10070.538544,9.313919
19663,4,3937,C2,10074.887951,NaN
19664,4,3938,C2,10075.039008,NaN
19665,4,3939,C2,10077.284456,NaN


In [20]:
results["run"]

,category,n_patients,mean_response_time,run,mean_utilisation
0,C1,184,8.157759,0,NaN
1,C2,2180,9.863232,0,NaN
2,C3,1237,12.069095,0,NaN
3,C4,292,11.776244,0,NaN
4,all,<NA>,NaN,0,0.126790
5,C1,166,8.680097,1,NaN
6,C2,2092,10.067462,1,NaN
7,C3,1400,12.006222,1,NaN
8,C4,287,12.317072,1,NaN
9,all,<NA>,NaN,1,0.129804


In [21]:
results["overall"]

,category,mean_n_patients,mean_response_time,mean_utilisation
0,C1,176.4,8.322830,NaN
1,C2,2185.6,10.011374,NaN
2,C3,1285.4,12.084111,NaN
3,C4,286.0,11.956038,NaN
4,all,NaN,NaN,0.128344


In [22]:
config.n_ambulances = 1
config.data_collection_period = 50_000
config.log_to_console = False
runner = Runner(config)
results = runner.run_single(run_number=0)

/home/amy/Documents/ambulance/ambdes/src/ambdes/results.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([summary, utilisation], ignore_index=True)


In [23]:
results["run"]

,category,n_patients,mean_response_time,run,mean_utilisation
0,C1,913,21336.868256,0,NaN
1,C2,10697,24713.270772,0,NaN
2,C3,6077,24027.830165,0,NaN
3,C4,1369,25178.875718,0,NaN
4,all,<NA>,NaN,0,0.999961


In [24]:
results["patients"].head(20)

,run,patient_id,category,call_timestamp,response_time
0,0,1,C3,1.965099,7.500451
1,0,2,C2,2.783496,127.094760
2,0,3,C3,3.429820,207.278225
3,0,4,C2,7.624019,316.133111
4,0,5,C3,8.305273,412.828393
5,0,6,C3,9.043671,538.852687
6,0,7,C2,9.632318,652.876531
7,0,8,C3,11.309869,750.446268
8,0,9,C2,11.630023,861.294090
9,0,10,C4,12.729495,969.036346


In [25]:
results["model"].logger.to_dataframe().head(30)

,entity_id,event_type,event,time,run_number,resource_id
0,1,arrival_departure,arrival,1.965099,0,NaN
1,1,queue,ambulance_wait_begins,1.965099,0,NaN
2,1,resource_use,ambulance_assigned,1.965099,0,1.0
3,2,arrival_departure,arrival,2.783496,0,NaN
4,2,queue,ambulance_wait_begins,2.783496,0,NaN
5,3,arrival_departure,arrival,3.429820,0,NaN
6,3,queue,ambulance_wait_begins,3.429820,0,NaN
7,4,arrival_departure,arrival,7.624019,0,NaN
8,4,queue,ambulance_wait_begins,7.624019,0,NaN
9,5,arrival_departure,arrival,8.305273,0,NaN
